In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")


In [2]:
# Đọc dữ liệu 311 Cases, chỉ lấy loại yêu cầu bảo trì cây (Tree Maintenance)
url_cases = "https://data.sfgov.org/resource/vw6y-z8j6.csv?$where=service_name=%27Tree%20Maintenance%27&$limit=500000"
df_cases = pd.read_csv(url_cases)
print(f"Number of rows    : {df_cases.shape[0]:,}")
print(f"Number of columns : {df_cases.shape[1]}")


Number of rows    : 175,814
Number of columns : 25


In [3]:
# kiểm tra thông tin dữ liệu
print(df_cases.info())  # Kiểm tra kiểu dữ liệu
print(df_cases.head())  # Xem trước 5 dòng đầu


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175814 entries, 0 to 175813
Data columns (total 25 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   service_request_id               175814 non-null  int64  
 1   requested_datetime               175814 non-null  object 
 2   closed_date                      148011 non-null  object 
 3   updated_datetime                 175814 non-null  object 
 4   status_description               175814 non-null  object 
 5   status_notes                     175703 non-null  object 
 6   agency_responsible               175814 non-null  object 
 7   service_name                     175814 non-null  object 
 8   service_subtype                  175814 non-null  object 
 9   service_details                  175572 non-null  object 
 10  address                          175813 non-null  object 
 11  street                           174447 non-null  object 
 12  su

In [4]:
# Đổi tên cột khu phố và chuyển cột thời gian yêu cầu về dạng datetime
df_cases = df_cases.rename(columns={"analysis_neighborhood": "nbhd"})
df_cases["requested_datetime"] = pd.to_datetime(df_cases["requested_datetime"], errors="coerce")
df_cases[["nbhd", "requested_datetime", "service_name"]].head()


,nbhd,requested_datetime,service_name
0,Inner Richmond,2021-01-19 06:52:00,Tree Maintenance
1,Lone Mountain/USF,2021-02-04 10:36:00,Tree Maintenance
2,North Beach,2021-01-22 18:00:00,Tree Maintenance
3,Hayes Valley,2021-01-19 10:44:00,Tree Maintenance
4,Mission,2021-01-23 16:08:00,Tree Maintenance


In [5]:
# kiểm tra dữ liệu thiếu
missing_values = df_cases.isnull().sum()
missing_percentage = (df_cases.isnull().mean() * 100)
missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage (%)': missing_percentage
})

missing_df = missing_df[missing_df['Missing Values'] > 0].sort_values(by='Percentage (%)', ascending=False)
print(missing_df)


                                 Missing Values  Percentage (%)
media_url                                118707       67.518514
bos_2012                                  44280       25.185708
closed_date                               27803       15.813871
nbhd                                      11769        6.694006
police_district                            1421        0.808241
street                                     1367        0.777526
neighborhoods_sffind_boundaries            1346        0.765582
supervisor_district                        1262        0.717804
source                                      251        0.142765
service_details                             242        0.137645
status_notes                                111        0.063135
point                                         6        0.003413
lat                                           6        0.003413
long                                          6        0.003413
point_geom                              

In [6]:
# Bỏ các dòng không xác định được khu phố hoặc thời gian yêu cầu (không thể dùng cho chuỗi thời gian theo khu phố)
n0 = len(df_cases)
df_cases = df_cases.dropna(subset=["nbhd", "requested_datetime"])
print(f"Đã bỏ {n0 - len(df_cases)} dòng thiếu khu phố hoặc thời gian yêu cầu")


Đã bỏ 11769 dòng thiếu khu phố hoặc thời gian yêu cầu


In [7]:
# Kiểm tra trùng lặp
duplicates = df_cases.duplicated().sum()
print("Duplicate rows:", duplicates)

df_cases = df_cases.drop_duplicates()


Duplicate rows: 0


In [8]:
# Tổng hợp số ca yêu cầu bảo trì cây thành chuỗi theo từng khu phố - ngày
df_cases["date"] = df_cases["requested_datetime"].dt.floor("D")
daily = df_cases.groupby(["nbhd", "date"]).size().rename("n").reset_index()

print(f"Số dòng sau khi tổng hợp : {daily.shape[0]:,}")
print(f"Số khu phố               : {daily['nbhd'].nunique()}")


Số dòng sau khi tổng hợp : 88,402
Số khu phố               : 41


In [9]:
# Lấp đầy lưới thời gian: mỗi khu phố phải có đủ mọi ngày trong khoảng dữ liệu (ngày không có ca thì n = 0)
full_idx = pd.MultiIndex.from_product(
    [daily["nbhd"].unique(), pd.date_range(daily["date"].min(), daily["date"].max(), freq="D")],
    names=["nbhd", "date"]
)
daily = daily.set_index(["nbhd", "date"]).reindex(full_idx, fill_value=0)["n"].reset_index()
print(f"Số dòng sau khi lấp đầy lưới thời gian: {daily.shape[0]:,}")


Số dòng sau khi lấp đầy lưới thời gian: 273,224


In [10]:
# Đọc dữ liệu SF Street-Use Permits (nguồn phụ chính: hoạt động thi công đường phố)
url_permits = "https://data.sfgov.org/resource/b6tj-gt35.csv?$where=permit_start_date>=%272015-01-01%27&$limit=500000"
df_permits = pd.read_csv(url_permits)
df_permits = df_permits.rename(columns={"analysis_neighborhood": "nbhd"})
print(f"Number of rows    : {df_permits.shape[0]:,}")
print(f"Number of columns : {df_permits.shape[1]}")


Number of rows    : 446,776
Number of columns : 31


In [11]:
# Chuyển đổi ngày cấp phép / hết hạn, loại bỏ lỗi nhập liệu (ví dụ ngày cấp phép rơi vào năm 2209)
df_permits["permit_start_date"] = pd.to_datetime(df_permits["permit_start_date"], errors="coerce")
df_permits["permit_end_date"] = pd.to_datetime(df_permits["permit_end_date"], errors="coerce")

n0 = len(df_permits)
df_permits = df_permits[df_permits["permit_start_date"] <= pd.Timestamp.now() + pd.Timedelta(days=365)]
print(f"Đã bỏ {n0 - len(df_permits)} dòng có ngày cấp phép bất thường (vượt xa tương lai)")

df_permits = df_permits.drop_duplicates(subset=["permit_number", "nbhd"])


Đã bỏ 8 dòng có ngày cấp phép bất thường (vượt xa tương lai)


In [12]:
# Xây dựng biến permits_active: số giấy phép đang hoạt động tại mỗi khu phố, mỗi ngày
# Chỉ tính những giấy phép có ĐẦY ĐỦ cả ngày bắt đầu lẫn ngày kết thúc (permit_end_date không rỗng),
# tránh trường hợp giấy phép "mồ côi" (thiếu ngày kết thúc) bị cộng dồn vĩnh viễn và làm sai lệch permits_active.
starts = df_permits.dropna(subset=["permit_start_date", "nbhd", "permit_end_date"])[["nbhd", "permit_start_date"]]
starts = starts.rename(columns={"permit_start_date": "date"})
starts["delta"] = 1

ends = df_permits.dropna(subset=["permit_end_date", "nbhd"])[["nbhd", "permit_end_date"]]
ends = ends.rename(columns={"permit_end_date": "date"})
ends["date"] = ends["date"] + pd.Timedelta(days=1)
ends["delta"] = -1

events = pd.concat([starts, ends])
events["date"] = events["date"].dt.floor("D")
events = events.groupby(["nbhd", "date"])["delta"].sum().reset_index()

full_idx2 = pd.MultiIndex.from_product(
    [events["nbhd"].unique(), pd.date_range(events["date"].min(), events["date"].max(), freq="D")],
    names=["nbhd", "date"]
)
permits_daily = events.set_index(["nbhd", "date"]).reindex(full_idx2, fill_value=0).reset_index()
permits_daily = permits_daily.sort_values(["nbhd", "date"])
permits_daily["permits_active"] = permits_daily.groupby("nbhd")["delta"].cumsum().clip(lower=0)

print(permits_daily["permits_active"].describe())

count    331444.000000
mean         45.839357
std          73.237389
min           0.000000
25%           1.000000
50%          12.000000
75%          65.000000
max         621.000000
Name: permits_active, dtype: float64


In [13]:
# Ghép permits_active vào bảng chính theo khu phố và ngày
daily = daily.drop(columns=["permits_active"], errors="ignore")  # xoá nếu đã có từ lần chạy trước, tránh trùng cột khi merge lại
daily = daily.merge(permits_daily[["nbhd", "date", "permits_active"]], on=["nbhd", "date"], how="left")

n_missing = daily["permits_active"].isna().sum()
daily["permits_active"] = daily["permits_active"].fillna(0)
print(f"Đã điền {n_missing} dòng không ghi nhận giấy phép nào bằng permits_active = 0")

Đã điền 96719 dòng không ghi nhận giấy phép nào bằng permits_active = 0


In [14]:
# Đọc dữ liệu thời tiết NOAA GHCN-Daily, trạm San Francisco (nguồn mở rộng)
url_weather = ("https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries"
               "&dataTypes=TMAX,TMIN,PRCP,AWND&stations=USW00023234"
               "&startDate=2015-01-01&endDate=2026-09-19&format=csv&units=metric")
df_weather = pd.read_csv(url_weather)
df_weather["DATE"] = pd.to_datetime(df_weather["DATE"], errors="coerce")
df_weather = df_weather.rename(columns={"DATE": "date"})
print(f"Number of rows    : {df_weather.shape[0]:,}")
df_weather[["date", "PRCP", "AWND", "TMAX"]].describe()


Number of rows    : 4,280


,date,PRCP,AWND,TMAX
count,4280,4280.000000,4278.000000,4260.000000
mean,2020-11-09 12:00:00,1.256846,4.638523,19.550329
min,2015-01-01 00:00:00,0.000000,0.400000,8.300000
25%,2017-12-05 18:00:00,0.000000,3.100000,16.100000
50%,2020-11-09 12:00:00,0.000000,4.600000,19.400000
75%,2023-10-15 06:00:00,0.000000,6.000000,22.200000
max,2026-09-19 00:00:00,102.100000,13.500000,40.000000
std,NaN,5.316583,2.016821,4.448236


In [15]:
# Ghép thời tiết vào bảng chính (thời tiết dùng chung cho toàn thành phố, không theo từng khu phố)
daily = daily.merge(df_weather[["date", "PRCP", "AWND", "TMAX"]], on="date", how="left")
print((daily[["PRCP", "AWND", "TMAX"]].isnull().mean() * 100).round(2))


PRCP    35.77
AWND    35.80
TMAX    36.07
dtype: float64


In [16]:
# Tạo đặc trưng thời gian và ép kiểu dữ liệu phân loại
daily["year"] = daily["date"].dt.year
daily["month"] = daily["date"].dt.month
daily["dow"] = daily["date"].dt.dayofweek
daily["nbhd"] = daily["nbhd"].astype("category")

daily.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 273224 entries, 0 to 273223
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   nbhd            273224 non-null  category      
 1   date            273224 non-null  datetime64[ns]
 2   n               273224 non-null  int64         
 3   permits_active  273224 non-null  float64       
 4   PRCP            175480 non-null  float64       
 5   AWND            175398 non-null  float64       
 6   TMAX            174660 non-null  float64       
 7   year            273224 non-null  int32         
 8   month           273224 non-null  int32         
 9   dow             273224 non-null  int32         
dtypes: category(1), datetime64[ns](1), float64(4), int32(3), int64(1)
memory usage: 15.9 MB


In [17]:
# Kiểm tra missing còn lại sau khi ghép nguồn phụ và nguồn mở rộng
missing = daily.isnull().sum()
missing = missing[missing > 0]
print(missing.sort_values(ascending=False))

# Fill missing cho các cột thời tiết (nếu có ngày thiếu dữ liệu trạm)
for col in ["PRCP", "AWND", "TMAX"]:
    daily[col] = daily[col].fillna(daily[col].median())


TMAX    98564
AWND    97826
PRCP    97744
dtype: int64


In [18]:
# kiểm tra dữ liệu sau khi làm sạch
print(daily.info())
print(f"Khoảng thời gian : {daily['date'].min().date()} đến {daily['date'].max().date()}")
print(f"Số khu phố       : {daily['nbhd'].nunique()}")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 273224 entries, 0 to 273223
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   nbhd            273224 non-null  category      
 1   date            273224 non-null  datetime64[ns]
 2   n               273224 non-null  int64         
 3   permits_active  273224 non-null  float64       
 4   PRCP            273224 non-null  float64       
 5   AWND            273224 non-null  float64       
 6   TMAX            273224 non-null  float64       
 7   year            273224 non-null  int32         
 8   month           273224 non-null  int32         
 9   dow             273224 non-null  int32         
dtypes: category(1), datetime64[ns](1), float64(4), int32(3), int64(1)
memory usage: 15.9 MB
None
Khoảng thời gian : 2008-07-01 đến 2026-09-28
Số khu phố       : 41


In [21]:
daily.to_csv("SF_Tree_Maintenance_Daily_Data_Cleaned.csv", index=False)
print("Dữ liệu đã được làm sạch và lưu thành công!")


Dữ liệu đã được làm sạch và lưu thành công!
